# Lecture 14.1 demo: from policy gradients to RLHF, DPO and GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/unreal-kz/lecture-14-demo-AI-course/blob/main/lecture14_policy_llm_rl.ipynb)

The toys behind the lecture, in code you can change. **Nothing to install, no key, no GPU**: it needs numpy only and
runs in well under a minute. Nothing here is graded and nothing is handed in.

Run the cells in order. A **✏️ cell** is one you are meant to edit. Every number about these toys on the slides came
from this code with the seeds below. Everything is synthetic: a firm that does not exist and a "language model" with
five possible answers. It shows how the methods behave, not how large models behave.

In [ ]:
#@title The toys (policy_toys.py, pasted unchanged; you only need to run this cell)
"""Small, exact toys behind Lecture 14.1: policy gradients, PPO clipping, preferences, DPO and GRPO.

The module is the single source of every number in the lecture. It needs numpy only. Everything is synthetic.

Part 1 learns a *policy* directly on the customer model of Lecture 12.1 (new, active, at risk; churned is the end).
Part 2 is one prompt with a handful of candidate answers: a reward model fitted to preferences, the effect of keeping
the trained model close to the supervised one (a KL penalty), direct preference optimisation, and group-relative
advantages with a rule-based reward.

There is deliberately no value iteration and no Q-learning here.
"""

from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass

import numpy as np

STATES = ("new", "active", "at risk")
GAMMA = 0.9
HORIZON = 60
END = 3  # churned

# margin (thousand tenge) of a month in each state under (nothing, discount)
REWARDS = np.array([[4.0, 2.0], [10.0, 7.0], [3.0, 1.0]])
# P[state, action, next] with next in (new, active, at risk, churned)
TRANSITIONS = np.array([
    [[0.0, 0.4, 0.3, 0.3], [0.0, 0.6, 0.3, 0.1]],
    [[0.0, 0.8, 0.2, 0.0], [0.0, 0.85, 0.15, 0.0]],
    [[0.0, 0.2, 0.4, 0.4], [0.0, 0.5, 0.4, 0.1]],
])


def sigmoid(x: np.ndarray | float) -> np.ndarray:
    """The logistic function."""
    return 1.0 / (1.0 + np.exp(-np.asarray(x, dtype=float)))


# ------------------------------------------------------------------------------------------------ Part 1: policies

def policy_value(p_discount: Sequence[float]) -> np.ndarray:
    """Exact value of the policy that offers a discount in state ``s`` with probability ``p_discount[s]``."""
    probs = np.asarray(p_discount, dtype=float)
    mix = np.stack([1 - probs, probs], axis=1)
    p_pi = np.einsum("sa,sat->st", mix, TRANSITIONS)[:, :3]
    r_pi = np.einsum("sa,sa->s", mix, REWARDS)
    return np.linalg.solve(np.eye(3) - GAMMA * p_pi, r_pi)


@dataclass(frozen=True)
class Batch:
    """Months from several customers followed with one fixed policy, flattened.

    Attributes:
        states: State of each month.
        actions: 0 nothing, 1 discount.
        returns: Discounted return from that month to the end of the customer's story.
        weights: ``gamma ** month`` (the month's position in its story).
        n_customers: Number of customers followed.
    """

    states: np.ndarray
    actions: np.ndarray
    returns: np.ndarray
    weights: np.ndarray
    n_customers: int


def sample_batch(theta: np.ndarray, n_customers: int, rng: np.random.Generator) -> Batch:
    """Follow ``n_customers`` new customers (until they churn or ``HORIZON`` months) with policy ``theta``."""
    probs = sigmoid(theta)
    cum = np.cumsum(TRANSITIONS, axis=2)
    cum[:, :, -1] = 1.0
    states, actions, returns, weights = [], [], [], []
    for _ in range(n_customers):
        s, episode = 0, []
        for _month in range(HORIZON):
            a = int(rng.random() < probs[s])
            nxt = int(np.searchsorted(cum[s, a], rng.random(), side="right"))
            episode.append((s, a, REWARDS[s, a]))
            if nxt == END:
                break
            s = nxt
        g = 0.0
        tail = []
        for s_t, a_t, r_t in reversed(episode):
            g = r_t + GAMMA * g
            tail.append((s_t, a_t, g))
        for t, (s_t, a_t, g_t) in enumerate(reversed(tail)):
            states.append(s_t)
            actions.append(a_t)
            returns.append(g_t)
            weights.append(GAMMA**t)
    return Batch(np.array(states), np.array(actions), np.array(returns), np.array(weights), n_customers)


def policy_gradient(theta: np.ndarray, batch: Batch, baseline: np.ndarray) -> np.ndarray:
    """REINFORCE gradient (Sutton and Barto, section 13.4): average of ``gamma^t (G_t - b(s_t)) grad log pi``."""
    probs = sigmoid(theta)
    score = batch.actions - probs[batch.states]  # d log pi(a|s) / d theta_s for a Bernoulli policy
    terms = batch.weights * (batch.returns - baseline[batch.states]) * score
    return np.bincount(batch.states, weights=terms, minlength=3) / batch.n_customers


def update_baseline(baseline: np.ndarray, batch: Batch, rate: float) -> np.ndarray:
    """Move the baseline of every visited state towards the mean return seen from it in this batch."""
    counts = np.bincount(batch.states, minlength=3)
    sums = np.bincount(batch.states, weights=batch.returns, minlength=3)
    seen = counts > 0
    new = baseline.copy()
    new[seen] += rate * (sums[seen] / counts[seen] - baseline[seen])
    return new


@dataclass(frozen=True)
class TrainRun:
    """Exact value of a new customer after every update, and the final discount probabilities."""

    values: np.ndarray
    p_discount: np.ndarray


def reinforce(seed: int | np.random.SeedSequence, n_updates: int, n_customers: int, lr: float, use_baseline: bool,
              baseline_rate: float = 0.2) -> TrainRun:
    """Train with REINFORCE; with ``use_baseline`` the returns are compared with a running estimate of ``v(s)``."""
    rng = np.random.default_rng(seed)
    theta = np.zeros(3)
    baseline = np.zeros(3)
    values = [float(policy_value(sigmoid(theta))[0])]
    for _ in range(n_updates):
        batch = sample_batch(theta, n_customers, rng)
        b = baseline if use_baseline else np.zeros(3)
        theta = theta + lr * policy_gradient(theta, batch, b)
        baseline = update_baseline(baseline, batch, baseline_rate)
        values.append(float(policy_value(sigmoid(theta))[0]))
    return TrainRun(np.array(values), sigmoid(theta))


def gradient_spread(n_batches: int, n_customers: int, seed: int | np.random.SeedSequence) -> dict[str, np.ndarray]:
    """Mean and standard deviation of the gradient estimate at the 50/50 policy, with and without a baseline.

    The baseline is the exact value of that policy, so the two estimators have the same expectation.
    """
    theta = np.zeros(3)
    exact_v = policy_value(sigmoid(theta))
    rng = np.random.default_rng(seed)
    plain, centred = [], []
    for _ in range(n_batches):
        batch = sample_batch(theta, n_customers, rng)
        plain.append(policy_gradient(theta, batch, np.zeros(3)))
        centred.append(policy_gradient(theta, batch, exact_v))
    return {"plain_mean": np.mean(plain, axis=0), "plain_sd": np.std(plain, axis=0, ddof=1),
            "base_mean": np.mean(centred, axis=0), "base_sd": np.std(centred, axis=0, ddof=1)}


# ------------------------------------------------------------------------------------------------ PPO

def surrogate_steps(theta_old: np.ndarray, batch: Batch, advantages: np.ndarray, lr: float, n_steps: int,
                    clip: float | None) -> np.ndarray:
    """``n_steps`` gradient steps on one batch of the surrogate ``mean(ratio * A)``, optionally clipped as in PPO.

    PPO's clipped objective is ``mean(min(r A, clip(r, 1-eps, 1+eps) A))``; its gradient is that of ``r A`` where the
    unclipped term is the smaller one, and zero elsewhere.
    """
    p_old = sigmoid(theta_old)
    prob_old = np.where(batch.actions == 1, p_old[batch.states], 1 - p_old[batch.states])
    theta = theta_old.astype(float).copy()
    for _ in range(n_steps):
        p = sigmoid(theta)
        prob = np.where(batch.actions == 1, p[batch.states], 1 - p[batch.states])
        ratio = prob / prob_old
        active = np.ones_like(ratio, dtype=bool)
        if clip is not None:
            active = ratio * advantages <= np.clip(ratio, 1 - clip, 1 + clip) * advantages
        grad_ratio = ratio * (batch.actions - p[batch.states])
        terms = np.where(active, advantages * grad_ratio, 0.0)
        theta = theta + lr * np.bincount(batch.states, weights=terms, minlength=3) / len(ratio)
    return theta


def max_ratio(theta_old: np.ndarray, theta_new: np.ndarray) -> float:
    """Largest factor by which the probability of some action in some state changed (up or down)."""
    p0, p1 = sigmoid(theta_old), sigmoid(theta_new)
    ratios = np.concatenate([p1 / p0, (1 - p1) / (1 - p0)])
    return float(np.max(np.maximum(ratios, 1 / ratios)))


# ------------------------------------------------------------------------------------------------ Part 2: preferences

@dataclass(frozen=True)
class Answer:
    """One candidate answer to a fixed question: whether it is right and how long it is (1 = short)."""

    name: str
    correct: int
    length: int
    p_sft: float


ANSWERS: tuple[Answer, ...] = (
    Answer("short, right", 1, 1, 0.30),
    Answer("short, wrong", 0, 1, 0.30),
    Answer("long, right", 1, 2, 0.20),
    Answer("long, wrong", 0, 2, 0.19),
    Answer("very long, wrong", 0, 6, 0.01),
)


def sft_probs() -> np.ndarray:
    """How often the supervised model gives each candidate answer."""
    p = np.array([a.p_sft for a in ANSWERS])
    return p / p.sum()


def true_utility() -> np.ndarray:
    """What the firm wants: right answers. Length is worth nothing."""
    return np.array([2.0 * a.correct for a in ANSWERS])


def features() -> np.ndarray:
    """Columns: correct, length."""
    return np.array([[a.correct, a.length] for a in ANSWERS], dtype=float)


LENGTH_CAP = 2  # labelers like longer answers, but only up to "long"
SEEN = 4  # the common answers; the very long one never reaches a labeler


def labeler_utility(length_bias: float) -> np.ndarray:
    """What labelers reward: right answers and, with ``length_bias``, longer ones up to ``LENGTH_CAP``."""
    return true_utility() + length_bias * np.minimum(features()[:, 1], LENGTH_CAP)


def sample_preferences(n_pairs: int, seed: int | np.random.SeedSequence, length_bias: float) -> np.ndarray:
    """``(winner, loser)`` index pairs among the ``SEEN`` common answers, drawn as often as the model writes them.

    A pair is judged by the Bradley-Terry model: ``Pr(i beats j) = sigmoid(u_i - u_j)``.
    """
    rng = np.random.default_rng(seed)
    utility = labeler_utility(length_bias)
    p = sft_probs()[:SEEN] / sft_probs()[:SEEN].sum()
    pairs: list[tuple[int, int]] = []
    while len(pairs) < n_pairs:
        i, j = rng.choice(SEEN, size=2, p=p)
        if i == j:
            continue
        if rng.random() < sigmoid(utility[i] - utility[j]):
            pairs.append((i, j))
        else:
            pairs.append((j, i))
    return np.array(pairs)


def fit_reward_model(pairs: np.ndarray, x: np.ndarray, steps: int = 4000, lr: float = 0.5) -> np.ndarray:
    """Weights ``w`` maximising the Bradley-Terry likelihood of the preferences with scores ``x @ w``."""
    diff = x[pairs[:, 0]] - x[pairs[:, 1]]
    w = np.zeros(x.shape[1])
    for _ in range(steps):
        w = w + lr * diff.T @ (1.0 - sigmoid(diff @ w)) / len(pairs)
    return w


def kl(p: np.ndarray, q: np.ndarray) -> float:
    """``KL(p || q)`` in nats."""
    mask = p > 0
    return float(np.sum(p[mask] * np.log(p[mask] / q[mask])))


def kl_regularised(reward: np.ndarray, beta: float) -> np.ndarray:
    """The distribution maximising ``E_p[reward] - beta * KL(p || p_sft)``: ``p_sft * exp(reward / beta)``, normalised."""
    logits = np.log(sft_probs()) + reward / beta
    logits -= logits.max()
    p = np.exp(logits)
    return p / p.sum()


def dpo(pairs: np.ndarray, beta: float, steps: int = 20000, lr: float = 2.0) -> np.ndarray:
    """Minimise the DPO loss over a free policy on the candidates, starting from the supervised model."""
    ref = np.log(sft_probs())
    theta = ref.copy()
    for _ in range(steps):
        logp = theta - np.log(np.sum(np.exp(theta)))
        margin = beta * (logp - ref)
        delta = margin[pairs[:, 0]] - margin[pairs[:, 1]]
        weight = beta * (1.0 - sigmoid(delta)) / len(pairs)
        grad_logp = np.zeros(len(theta))
        np.add.at(grad_logp, pairs[:, 0], weight)
        np.add.at(grad_logp, pairs[:, 1], -weight)
        grad = grad_logp - grad_logp.sum() * np.exp(logp)  # chain rule through the softmax
        theta = theta + lr * grad
    p = np.exp(theta - theta.max())
    return p / p.sum()


def tabular_reward(pairs: np.ndarray) -> np.ndarray:
    """A reward model with one free score per candidate, fitted to the preferences (the two-step route)."""
    return fit_reward_model(pairs, np.eye(len(ANSWERS)), steps=20000, lr=2.0)


# ------------------------------------------------------------------------------------------------ GRPO

def group_advantages(rewards: Sequence[float]) -> np.ndarray:
    """GRPO's advantage within one group of answers to the same question: ``(r - mean) / std`` (zero if all equal)."""
    r = np.asarray(rewards, dtype=float)
    sd = r.std()
    if sd == 0:
        return np.zeros_like(r)
    return (r - r.mean()) / sd


def grpo_train(seed: int | np.random.SeedSequence, n_steps: int, group: int, lr: float, p_start: Sequence[float],
               correct: int) -> np.ndarray:
    """Probability of the correct answer after every step, training with a rule-based reward (1 if right, else 0).

    A policy-gradient update with group-relative advantages and no value model. The full method also keeps a clipped
    ratio and a KL penalty; with one gradient step per group the ratio is 1, so they are left out of this toy.
    """
    rng = np.random.default_rng(seed)
    theta = np.log(np.asarray(p_start, dtype=float))
    curve = []
    for _ in range(n_steps + 1):
        p = np.exp(theta - theta.max())
        p /= p.sum()
        curve.append(p[correct])
        answers = rng.choice(len(p), size=group, p=p)
        adv = group_advantages((answers == correct).astype(float))
        grad = np.zeros_like(theta)
        for a, g in zip(answers, adv, strict=True):
            grad += g * (np.eye(len(p))[a] - p)
        theta = theta + lr * grad / group
    return np.array(curve)


def lora_counts(d_in: int, d_out: int, rank: int) -> dict[str, int]:
    """Trainable numbers of one weight matrix: all of it, or two low-rank factors."""
    return {"full": d_in * d_out, "lora": rank * (d_in + d_out)}

## 1. One customer, seen by REINFORCE (slide 5)

The policy offers a discount with probability one half. For each month: the return that followed, the baseline (the
value of the state) and which way the update pushes the probability of a discount.

In [ ]:
import numpy as np

theta = np.zeros(3)  # one number per state: P(discount) = sigmoid(theta)
v_half = policy_value(sigmoid(theta))
batch = sample_batch(theta, 1, np.random.default_rng(6))
for month, (s, a, g) in enumerate(zip(batch.states, batch.actions, batch.returns), start=1):
    adv = g - v_half[s]
    push = "up" if (a - 0.5) * adv > 0 else "down"
    print(f"{month}  {STATES[s]:>8}  {'discount' if a else 'nothing ':8}  return {g:5.1f}  baseline {v_half[s]:5.1f}"
          f"  advantage {adv:6.1f}  push discount {push}")

## 2. What a baseline does to the noise (slide 6)

In [ ]:
spread = gradient_spread(4000, 10, np.random.SeedSequence([15]))
for key, label in (("plain", "no baseline"), ("base", "baseline")):
    print(f"{label:>11}: mean {np.round(spread[key + '_mean'], 2)}  standard deviation {np.round(spread[key + '_sd'], 2)}")

## 3. REINFORCE, with and without the baseline (slide 7)

In [ ]:
for label, use in (("no baseline", False), ("baseline", True)):
    runs = [reinforce(np.random.SeedSequence([14, i]), 300, 10, 0.01, use) for i in range(20)]
    curves = np.array([run.values for run in runs])
    print(f"{label:>11}: mean value at 0/50/100/200/300 updates",
          " ".join(f"{curves[:, k].mean():.1f}" for k in (0, 50, 100, 200, 300)),
          "| worst run at 100:", f"{curves[:, 100].min():.1f}")
    print(f"{'':>11}  P(discount) at the end:", np.round(np.mean([run.p_discount for run in runs], axis=0), 2))

## 4. One small batch, many steps (slides 9 and 11)

Start from a policy that is already good, follow only 5 customers, then take 200 gradient steps on those 5 stories.

In [ ]:
def reuse(start_p, clip, n_batches=400):
    start = np.log(np.array(start_p) / (1 - np.array(start_p)))
    v = policy_value(sigmoid(start))
    after, shift = [], []
    for i in range(n_batches):
        b = sample_batch(start, 5, np.random.default_rng(np.random.SeedSequence([16, i])))
        new = surrogate_steps(start, b, b.returns - v[b.states], 0.1, 200, clip)
        after.append(policy_value(sigmoid(new))[0])
        shift.append(np.max(np.abs(sigmoid(new) - sigmoid(start))))
    after = np.array(after)
    return v[0], after.mean(), np.percentile(after, 5), np.median(shift)


for name, start_p in (("good start", (0.9, 0.1, 0.9)), ("50/50 start", (0.5, 0.5, 0.5))):
    for label, clip in (("no clip", None), ("clip 0.2", 0.2)):
        v0, mean, p5, shift = reuse(start_p, clip)
        print(f"{name:>11} {label:>8}: value {v0:.1f} -> mean {mean:.1f}, 5th percentile {p5:.1f}, "
              f"typical largest shift {100 * shift:.0f} points")

✏️ Try other clip values. A very large one behaves like no clipping.

In [ ]:
CLIP_TRY = 0.2  # try 0.05, 0.5, 2.0
v0, mean, p5, shift = reuse((0.9, 0.1, 0.9), CLIP_TRY, n_batches=100)
print(f"clip {CLIP_TRY}: mean {mean:.1f}, 5th percentile {p5:.1f}, typical largest shift {100 * shift:.0f} points")

## 5. A reward model from comparisons (slide 16)

Labelers compare two answers at a time. They like right answers and, a little, longer ones. The very long answer is
so rare that nobody ever compares it.

In [ ]:
x = features()
pairs = sample_preferences(2000, np.random.SeedSequence([17, 0]), 0.8)
w = fit_reward_model(pairs, x)
held_out = sample_preferences(2000, np.random.SeedSequence([17, 1]), 0.8)
score = x @ w
print("weights (right, length):", np.round(w, 2))
print(f"agreement with held-out choices: {np.mean(score[held_out[:, 0]] > score[held_out[:, 1]]):.1%}")
for a, s, u in zip(ANSWERS, score, labeler_utility(0.8)):
    print(f"{a.name:>17}: reward model {s:5.2f}   labelers {u:4.1f}")

## 6. Optimising against it, with a KL penalty (slide 18)

The answer distribution that maximises the reward model's score minus `beta` times the distance from the supervised
model. A small `beta` lets it move far.

In [ ]:
print("  beta  reward model  labelers  P(right)     KL  P(very long, wrong)")
for beta in (None, 10, 3, 1, 0.5, 0.3, 0.2, 0.1):
    p = sft_probs() if beta is None else kl_regularised(score, beta)
    print(f"{'SFT' if beta is None else beta:>6}  {p @ score:12.2f}  {p @ labeler_utility(0.8):8.2f}"
          f"  {p @ (true_utility() > 0):8.1%}  {kl(p, sft_probs()):5.2f}  {p[4]:19.1%}")

## 7. DPO: the same answer with one loss (slide 21)

In [ ]:
two_step = kl_regularised(tabular_reward(pairs), 1.0)
direct = dpo(pairs, 1.0)
for a, p0, p1, p2 in zip(ANSWERS, sft_probs(), two_step, direct):
    print(f"{a.name:>17}: SFT {p0:5.1%}  reward model then optimise {p1:5.1%}  DPO {p2:5.1%}")
print(f"largest difference: {np.abs(two_step - direct).max():.0e}")

## 8. GRPO: compare answers within a group (slides 22 and 23)

The question has one right answer among four; the model starts with a 10 % chance of giving it. Reward 1 if right.

In [ ]:
print("advantages for rewards 1,0,0,1,0,0,0,0:", np.round(group_advantages([1, 0, 0, 1, 0, 0, 0, 0]), 2))
for group in (2, 8, 32):
    curves = np.array([grpo_train(np.random.SeedSequence([18, group, i]), 200, group, 0.5, (0.1, 0.3, 0.3, 0.3), 0)
                       for i in range(50)])
    print(f"group {group:>2}: P(right) after 0/10/50/100/200 steps",
          " ".join(f"{curves[:, k].mean():.2f}" for k in (0, 10, 50, 100, 200)),
          f"| groups with nothing to learn at the start: {0.9**group + 0.1**group:.0%}")

✏️ Make the question harder: how long does it take if the model is almost never right at the start?

In [ ]:
P_RIGHT = 0.01  # the chance of the right answer at the start
GROUP = 8
rest = (1 - P_RIGHT) / 3
curve = np.array([grpo_train(i, 200, GROUP, 0.5, (P_RIGHT, rest, rest, rest), 0) for i in range(20)]).mean(axis=0)
print("P(right) after 0/10/50/100/200 steps:", " ".join(f"{curve[k]:.2f}" for k in (0, 10, 50, 100, 200)))

## 9. LoRA arithmetic (slide 14)

In [ ]:
for rank in (4, 8, 16, 64):
    c = lora_counts(4096, 4096, rank)
    print(f"rank {rank:>2}: {c['lora']:>9,} trainable numbers instead of {c['full']:,} ({c['lora'] / c['full']:.2%})")